# 2HRX9P6HKXA8V

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Time Differences

In [ ]:
%store -r restaurant_data_unprocessed
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])

before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

In [ ]:
loc_id = '2HRX9P6HKXA8V'

In [ ]:
sales_and_menu_data[loc_id].index

In [ ]:
time_differences_details[loc_id]

10 Hour Difference

In [ ]:
sales_and_menu_data[restaurants_by_4m_coverage[0]]['item_name'].value_counts().sort_values(ascending=False).head(10)

In [ ]:
sales_and_menu_data[restaurants_by_4m_coverage[0]].loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

20 Hour Difference

In [ ]:
sales_and_menu_data[restaurants_by_4m_coverage[0]].loc[pd.Timestamp('2020-06-07 22:27:20+00:00'):pd.Timestamp('2020-06-09').tz_localize('UTC')].head(10)

In [ ]:
time_differences['2HRX9P6HKXA8V']

In [ ]:
time_differences_details['2HRX9P6HKXA8V'][0][time_differences_details['2HRX9P6HKXA8V'][0] == 5]

In [ ]:
sales_and_menu_data['2HRX9P6HKXA8V'].loc['2020-08-24 4:55:15+00:00':'2020-08-25 10:20:15+00:00']